In [ ]:
# Di cell pertama notebook, sebelum import apapun
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    force=True  # ← ini yang penting, override existing handlers
)

In [ ]:
from src.graph.workflow import build_rag_graph
from src.graph.state import RAGState
from src.pageindex.utils import load_toc

app = build_rag_graph()
page_index_result = load_toc("index.json")

initial_state = RAGState(
    query="siapa handoko?",
    structure=page_index_result["structure"],
    visited_ids=[],
    gathered_texts=[],
    gathered_titles=[],
    is_sufficient=False,
    missing_info="",
    iterations=0,
    early_stop=False,
    answer="",
    chat_history=[],
    _pending_node_ids=[],
)

result = await app.ainvoke(initial_state)
print("Answer:", result["answer"])
print("From:", result["gathered_titles"])

c:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-20 23:11:34,982 - src.pageindex.search.tree_search - INFO - table_of_contents: [
  {
    "title": "Daftar pengurus ICCN antara lain :",
    "start_index": 1,
    "end_index": 1,
    "nodes": [
      {
        "title": "Dewan Pengawas (Supervisory Board) ICCN :",
        "start_index": 2,
        "end_index": 2,
        "node_id": "0001",
        "summary": "The partial document outlines the composition of the Dewan Pengawas (Supervisory Board) of ICCN, specifying the roles and names of its members. It identifies PAULUS MINTARGA as the Ketua (Chairman) of the Dewan Pengawas ICCN, and LILIEK SETIAWAN and GREGORIUS SRI WURYANTO as Anggota (Members) of the Dewan Pengawas ICCN."
      },
      {
        "title": "Dewan P

Membaca Node: [0004] Dewan Pakar (Expert Council) ICCN :


2026-03-20 23:11:40,576 - langchain_aws.chat_models.bedrock_converse - INFO - Using Bedrock Converse API to generate response


Info ditemukan: Menambahkan ke Knowledge Stack.
Evaluasi -> Sufficient: True | Missing: nothing
Answer: [{'type': 'text', 'text': 'Based on the provided document context, Handoko Hendroyo is identified as the Ketua Dewan Pakar (Chairman of the Expert Council) for the areas of Ciptaruang, Kekayaan Intelektual (HKI), Keterlacakan, and City Branding at ICCN (Indonesian Chamber of Commerce and Industry or similar organization). \n\nThere is no additional personal information or background details provided about Handoko Hendroyo beyond his role and responsibilities within ICCN.', 'index': 0}]
From: ['Dewan Pakar (Expert Council) ICCN :']


: 

In [2]:
from docling_extractor import extract_single_document

result = extract_single_document(
    input_path="SK36_PENGURUS_ICCN.pdf",
    output_dir="/output",
    document_id="test_doc_001"
)

# Build markdown from sections
md_lines = []

for s in result["sections"]:
    text = s.get("content_text", "").strip()
    if not text:
        continue
    
    section_type = s.get("section_type", "paragraph")
    heading_level = s.get("heading_level")
    
    if section_type in ("title",):
        md_lines.append(f"# {text}\n")
    elif section_type == "heading" and heading_level:
        prefix = "#" * min(heading_level, 6)
        md_lines.append(f"{prefix} {text}\n")
    elif section_type in ("list_item",):
        md_lines.append(f"- {text}")
    elif section_type in ("caption",):
        md_lines.append(f"*{text}*\n")
    elif section_type in ("footnote",):
        md_lines.append(f"> {text}\n")
    else:
        md_lines.append(f"{text}\n")

# Insert tables at the right page position
for t in result["tables"]:
    md = t.get("markdown", "") or t.get("markdown_content", "")
    if md:
        md_lines.append(f"\n{md}\n")

# Write to file
output_md = "\n".join(md_lines)
with open("/output/result.md", "w", encoding="utf-8") as f:
    f.write(output_md)

print("Saved to /output/result.md")
# print(output_md[:500])  # preview

Saved to /output/result.md


In [ ]:
from docling_extractor import DoclingExtractor
import os

# Gunakan "./output" (relative path) agar folder dibuat di tempat script berjalan.
# Jika "/output", sistem mungkin akan mencoba membuat folder di root directory 
# yang bisa menyebabkan PermissionError (tergantung OS Anda).
output_dir = "./output" 
doc_id = "test_doc_001"

# Panggil Docling langsung, tanpa multiprocessing wrapper
extractor = DoclingExtractor(output_dir=output_dir)
result = extractor.extract("MCP.pdf", doc_id)

print("Tools used:", result["tools_used"])
print("Sections:", len(result["sections"]))
print("Tables:", len(result["tables"]))

# 1. Pastikan folder tujuannya sudah dibuat
folder_path = os.path.join(output_dir, doc_id)
os.makedirs(folder_path, exist_ok=True) 

# 2. Tentukan lokasi lengkap file .md
md_path = os.path.join(folder_path, f"{doc_id}.md")

# 3. Ekstrak teks markdown dari hasil 
# PENTING: Jika script mencetak "Tidak ada teks markdown...", coba cek apa saja
# key yang ada di dalam `result` dengan menambahkan: print(result.keys())
# Lalu ganti "markdown_text" di bawah dengan nama key yang sesuai (misal "text" atau "markdown").
markdown_content = result.get("markdown_text", "") 

# 4. Simpan ke dalam file
if markdown_content:
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)
    print(f"Berhasil! File markdown disimpan di: {md_path}")
else:
    print("Tidak ada teks markdown yang ditemukan untuk disimpan.")

: 

In [ ]:
import os
import json
import asyncio
import nest_asyncio
from pathlib import Path
from src.services.llm_service import get_llm
from src.pageindex.indexer.page_index_md import md_to_tree

nest_asyncio.apply()

# ── Init LLM ──
llm = get_llm(
    model_id="amazon.nova-pro-v1:0",
    max_tokens=4096,
    streaming=False,
    temperature=0.0
)

# ── Run ──
result = asyncio.run(md_to_tree(
    md_path="pengurus_iccn.md",
    if_thinning=True,
    min_token_threshold=1000,
    if_add_node_summary='yes',
    summary_token_threshold=500,
    if_add_doc_description='yes',
    if_add_node_id='yes',
    model=llm,
))

# ── Save ──
os.makedirs("output", exist_ok=True)
output_path = "output/pengurus_iccn_structure.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"PageIndex saved to: {output_path}")

# ── Preview ──
print("Doc description:", result.get("doc_description", ""))
print("\nTop-level nodes:")
for node in result.get("structure", []):
    print(f"  [{node.get('node_id')}] {node['title']}")
    for child in node.get("nodes", []):
        print(f"    [{child.get('node_id')}] {child['title']}")

Extracting nodes from markdown...
Extracting text content from nodes...
Thinning nodes...
Building tree from nodes...
Formatting tree structure...
Generating summaries for each node...
Generating document description...
PageIndex saved to: output/pengurus_iccn_structure.json
Doc description: This document is a decision letter from the Management Board of the Indonesian Creative Cities Network, detailing the organizational structure and leadership for the period 2025-2028, including various boards, councils, and directorates.

Top-level nodes:
  [0000] SURAT KEPUTUSAN DEWAN PENGURUS PERKUMPULAN JEJARING KOTA DAN KABUPATEN KREATIF DI INDONESIA
    [0001] Nomor: 36/SK-ICCN/XII/2025
    [0002] Tentang: SUSUNAN PENGURUS PERKUMPULAN JEJARING KOTA DAN KABUPATEN KREATIF DI INDONESIA PERIODE TAHUN 2025-2028
    [0003] Menimbang
    [0004] Memperhatikan
    [0005] Menetapkan
    [0006] Memutuskan
    [0007] Susunan Pengurus ICCN Periode 2025-2028
    [0008] Deputi 1 - Tata Kelola, Partisipasi, d

In [3]:
import json

with open("output/pengurus_iccn_structure.json", encoding="utf-8") as f:
    result = json.load(f)

# Cek node yang berisi tabel
for node in result["structure"]:
    text = node.get("text", "") or node.get("summary", "")
    if "|" in text:  # tabel markdown pakai pipe
        print(f"[{node['node_id']}] {node['title']}")
        print(text[:500])
        print("---")